# Physical moves: chess move -> arm primitive steps

**Deliverable:** physical move execution for every special move (README, "Known limitations":
*captures, castling, en passant, and promotion need their own sequences*).

A legal `chess.Move` on a `chess.Board` becomes an ordered list of arm primitives. The arm side
(`gambit_arm`) implements the primitives; this notebook only defines the protocol and the plans.

| Move | Steps |
|---|---|
| normal | `move_piece(from, to)` |
| capture | `capture_to_graveyard(to)`, `move_piece(from, to)` (clear the target first) |
| castling | `move_piece(king_from, king_to)`, `move_piece(rook_from, rook_to)` (king first, FIDE touch rule) |
| en passant | `capture_to_graveyard(captured_pawn_square)` (not the target square!), `move_piece(from, to)` |
| promotion | `capture_to_graveyard(from)` (pawn leaves), `place_from_graveyard(new_piece, to)` |
| capture + promotion | `capture_to_graveyard(to)`, `capture_to_graveyard(from)`, `place_from_graveyard(new_piece, to)` |

Promotion pieces come from the graveyard: the spare queens (`pieces.spare_queens` in
`gambit_bringup/config/board.yaml`) plus everything captured so far. `Graveyard` tracks that inventory.

**TODO checklist**
- [ ] `Graveyard` inventory, `Step.__str__`.
- [ ] Plans for every move type (`plan_physical_move` + castling helpers), verified by replaying
      them on a piece map (`apply_steps`).
- [ ] `execute_plan` against an `ArmInterface`.
- [ ] Self-checks (plan at the bottom).
- [ ] Decide what to do when the promotion piece is not in the graveyard (e.g. underpromotion to
      a rook while both rooks are still on the board): ask the human, or use an upside-down rook
      as a queen... (raise `MissingPieceError` as a first version?).
- [ ] Decide graveyard slot layout (which slot holds which piece) together with `gambit_arm`.
- [ ] Optional: order steps to minimise arm travel (e.g. capture then move from the nearest side).

In [ ]:
from __future__ import annotations

from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Protocol

import chess
import yaml

BOARD_CFG_PATH = Path("../../gambit_bringup/config/board.yaml")
with open(BOARD_CFG_PATH) as _f:
    BOARD_CFG = yaml.safe_load(_f)
SPARE_QUEENS = BOARD_CFG["pieces"]["spare_queens"]

## Arm interface (implemented by `gambit_arm`, faked in `game_loop.ipynb`)

Squares are python-chess square indices (`chess.E4`). Each primitive returns `True` on success
(the arm side does its own grasp verification and retries; see `recovery.max_arm_retries`).

In [ ]:
class ArmInterface(Protocol):
    def pick(self, square: chess.Square) -> bool: ...
    def place(self, square: chess.Square) -> bool: ...
    def move_piece(self, from_sq: chess.Square, to_sq: chess.Square) -> bool: ...
    def capture_to_graveyard(self, square: chess.Square) -> bool: ...
    def place_from_graveyard(self, piece: chess.Piece, square: chess.Square) -> bool: ...
    def park(self) -> bool: ...

## Steps

A `Step` is one primitive call. `piece` is the piece that moves (or, for `place_from_graveyard`,
the piece taken out of the graveyard); it is carried along so the arm can choose a grasp height
from `pieces.height` in board.yaml and so the plan can be checked offline.

In [ ]:
MOVE, CAPTURE, FROM_GRAVEYARD = "move_piece", "capture_to_graveyard", "place_from_graveyard"


@dataclass(frozen=True)
class Step:
    op: str
    square: chess.Square                  # source square (move/capture) or target (from graveyard)
    to: Optional[chess.Square] = None     # target square for move_piece
    piece: Optional[chess.Piece] = None

    def __str__(self) -> str:
        """Readable form, e.g. 'move_piece(P e2->e4)', 'capture_to_graveyard(p d5)',
        'place_from_graveyard(Q -> e8)' (used in logs and assertion messages)."""
        # Design: one format per op; chess.square_name for squares, Piece.symbol() ("?" if None).
        # TODO: implement
        raise NotImplementedError


class MissingPieceError(RuntimeError):
    """The piece needed for a promotion is not in the graveyard."""

## Graveyard inventory

Starts with the spare queens; every capture adds the captured piece, every promotion removes the
promoted piece. (Pieces that left the board are the only ones the arm can bring back.)

In [ ]:
class Graveyard:
    def __init__(self, spare_queens: int = SPARE_QUEENS):
        """Inventory starts with `spare_queens` queens of each colour."""
        # Design: a collections.Counter keyed by chess.Piece in `self.pieces` (Piece is hashable).
        # TODO: implement
        raise NotImplementedError

    def add(self, piece: chess.Piece) -> None:
        """A piece went into the graveyard."""
        # TODO: implement
        raise NotImplementedError

    def take(self, piece: chess.Piece) -> None:
        """Remove one `piece`; raise MissingPieceError if there is none."""
        # TODO: implement
        raise NotImplementedError

    def has(self, piece: chess.Piece) -> bool:
        # TODO: implement
        raise NotImplementedError

    def copy(self) -> "Graveyard":
        """Independent copy (plans work on a copy and commit only on success)."""
        # TODO: implement
        raise NotImplementedError

## Planning

`plan_physical_move` reads the board *before* the move. python-chess encodes castling as the king
moving two squares (`e1g1`), so the rook squares are derived from the castling side. Chess960
castling (king "captures" own rook) can be handled by `board.is_castling` the same way.

In [ ]:
def castling_rook_squares(board: chess.Board, move: chess.Move) -> tuple[chess.Square, chess.Square]:
    """(rook_from, rook_to) for a castling move."""
    # Design:
    # - Side from board.is_kingside_castling / is_queenside_castling; rank from the king's square.
    # - Rook ends on the f-file (O-O) or d-file (O-O-O). It starts on the h/a file in standard
    #   chess; in Chess960 the move's to_square *is* the rook square (king "captures" its rook).
    # TODO: implement
    raise NotImplementedError


def king_castle_target(board: chess.Board, move: chess.Move) -> chess.Square:
    """Where the king physically ends: g-file (O-O) or c-file (O-O-O) on its own rank."""
    # TODO: implement
    raise NotImplementedError


def plan_physical_move(board: chess.Board, move: chess.Move,
                       graveyard: Optional[Graveyard] = None) -> list[Step]:
    """Ordered arm steps for a legal move. Updates `graveyard` (if given) as the plan would.

    Raises ValueError for an illegal move, MissingPieceError if a promotion piece is unavailable.
    """
    # Design:
    # - Reject moves not in board.legal_moves (legality is checked separately from placement).
    # - Plan against a *copy* of the graveyard and commit to the caller's object only when the
    #   whole plan succeeded, so a MissingPieceError leaves the inventory untouched.
    # - Branch with python-chess predicates: is_castling, is_en_passant, is_capture, move.promotion.
    #   Follow the table at the top: clear target squares before moving into them; en passant
    #   captures the pawn beside the target (same file as to_square, same rank as from_square).
    # - Promotion: the pawn goes to the graveyard and the new piece comes out of it.
    # - Every Step carries the piece involved (grasp height, offline checks).
    # TODO: implement
    raise NotImplementedError


def _commit(target: Optional[Graveyard], planned: Graveyard) -> None:
    """Copy the planned inventory into the caller's graveyard (if one was given)."""
    # TODO: implement
    raise NotImplementedError


def describe_plan(steps: list[Step]) -> str:
    """All steps as one readable string (for logs)."""
    # TODO: implement (join the str of each step)
    raise NotImplementedError

## Executing and replaying plans

`execute_plan` sends the steps to an `ArmInterface` and stops at the first failure (the game
manager then re-reads the board and goes to recovery). `apply_steps` replays steps on a plain
`{square: piece}` dict; it is how the fake arm in `game_loop.ipynb` moves pieces and how the
self-check can prove every plan produces the right position.

In [ ]:
@dataclass
class ExecutionResult:
    ok: bool
    steps_done: int
    failed_step: Optional[Step] = None


def execute_plan(arm: ArmInterface, steps: list[Step]) -> ExecutionResult:
    """Run the steps on the arm, stopping at the first primitive that returns False."""
    # Design:
    # - Dispatch each Step.op to the matching ArmInterface primitive; unknown op -> ValueError.
    # - Report how many steps completed and which one failed; never continue after a failure
    #   (the camera decides what the board looks like now, not the plan).
    # TODO: implement
    raise NotImplementedError


def apply_steps(piece_map: dict, steps: list[Step], graveyard: Optional[Counter] = None) -> dict:
    """Replay steps on a copy of a {square: Piece} map. Raises if a step is physically impossible."""
    # Design:
    # - Never mutate the input map; optionally count pieces sent to `graveyard`.
    # - Physically impossible steps should fail loudly: moving from an empty square, moving or
    #   placing onto an occupied square, capturing from an empty square.
    # TODO: implement
    raise NotImplementedError

## Self-check

For each special move: plan it, replay the plan on the board's piece map and compare with the
position python-chess computes after the move.

In [ ]:
# Self-check plan (write these as asserts once the functions above work):
# - Helper idea `_check(fen, uci, expected_ops, gy=None)`: plan, assert the op sequence, replay with
#   apply_steps on board.piece_map(), push the move and compare piece maps.
# - normal e2e4 -> [MOVE]; capture e4xd5 -> [CAPTURE, MOVE].
# - O-O white ("r3k2r/8/8/8/8/8/8/R3K2R w KQkq - 0 1", e1g1) -> king e1->g1 first, then rook h1->f1.
# - O-O-O black (same FEN with b to move, e8c8) -> rook a8->d8.
# - en passant ("4k3/8/8/3pP3/8/8/8/4K3 w - d6 0 2", e5d6) -> capture on d5, not d6.
# - promotion ("8/4P3/8/8/8/8/k7/4K3 w - - 0 1", e7e8q) -> [CAPTURE, FROM_GRAVEYARD]; one spare
#   white queen used, one white pawn added to the graveyard.
# - capture + promotion ("3r4/4P3/...", e7d8q) -> [CAPTURE, CAPTURE, FROM_GRAVEYARD]; underpromotion
#   to a rook works only if a white rook was captured earlier.
# - underpromotion to a knight with none available -> MissingPieceError, graveyard unchanged.
# - Replay a short game with castling and captures (e.g. "e4 e5 Nf3 Nc6 Bc4 Nf6 O-O Nxe4 Re1 d5
#   Bxd5 Qxd5 Nc3 Qa5 Rxe4+ Be7"): after each plan, the replayed map equals board.piece_map().
# Example shape of a test:
# b = chess.Board(); assert [s.op for s in plan_physical_move(b, chess.Move.from_uci("e2e4"))] == [MOVE]
print("Self-check: not written yet (see plan above)")